# The Euler equations with adaptive mesh refinement: a point explosion

This tutorial studies a scenario where a regular mesh is a poor fit:
a point explosion. A small region of high pressure in the middle of the
domain releases a shock wave which travels outwards as a thin ring. Most of
the domain remains at rest for most of the time, while the interesting
physics happens within a few volumes around the front. Such setups call
for adaptive mesh refinement (AMR), i.e. for a mesh which is fine where the
solution is steep and coarse everywhere else.

We first set up a baseline on a regular mesh. It gives us a reference
solution and shows where the regular mesh spends its effort. We run
through the following steps:

- how to set up the point explosion with the same Euler solver as in
  @ref tutorials_exahype2_10_euler;
- how to run and visualise the explosion;
- how the solution evolves, and why the first-order scheme smears out the
  shock;
- how many patches actually hold steep gradients, i.e. how much work a
  regular mesh wastes.

We assume that you have completed @ref tutorials_exahype2_10_euler, as we
reuse its flux and eigenvalue and skip most of the explanations of the
script's structure.

## The test case: a point explosion

We solve the compressible Euler equations in two dimensions over the square
$[-0.5,0.5]^2$. The gas is at rest everywhere. Within a circle of radius
$0.1$ around the origin, density and pressure are high. Outside, they are
low:

\begin{equation*}
(\rho, u_1, u_2, p)(x,y,0) =
\begin{cases}
(1, 0, 0, 1) & \text{if } \sqrt{x^2+y^2} < 0.1, \\
(0.125, 0, 0, 0.1) & \text{otherwise.}
\end{cases}
\end{equation*}

The states inside and outside are those of Sod's shock tube. Toro uses the
same states for his cylindrical explosion test, with a radius of $0.4$ on
the square $[-1,1]^2$ (E. F. Toro: Riemann Solvers and Numerical Methods
for Fluid Dynamics, Springer). We shrink the radius, so the explosion
starts from an almost point-like region.

Once the simulation starts, three waves emerge from the edge of the
circle:

- a shock which runs outwards and compresses the gas at rest;
- a contact discontinuity which follows the shock more slowly. It separates
  the gas from inside the circle from the gas that was outside;
- a rarefaction wave which runs inwards, reaches the centre and is
  reflected there. It leaves a region of low density behind.

There is no exact solution in closed form, but the setup is radially
symmetric. The solution has to stay radially symmetric, too, which is a
first sanity check of any simulation. With $\gamma=1.4$, the total energy
is $E = p/(\gamma-1)$, as the gas is at rest initially.

## Configuring the environment

As in the previous tutorials, we add the Python API to the search path:

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("../../src"))

import peano4.api
import exahype2

In [ ]:
try:
    import ipywidgets, matplotlib
except ImportError:
    !{sys.executable} -m pip install ipywidgets matplotlib

## Project and solver

The project and the solver are the same as in @ref tutorials_exahype2_10_euler.
Only the output directory changes. We need a flux, the maximum eigenvalue
and a point-wise initial condition.

In [ ]:
project = exahype2.api.Project(
    namespace    = ["tutorials", "exahype2", "euler"],
    project_name = "Euler",
    directory    = "13_euler-with-amr",
    executable   = "euler",
)

solver = exahype2.api.Solver(
    "EulerFV",
    4,   # unknowns
    0,   # auxiliary variables
)

solver.add_pde_term(exahype2.api.Flux(stateless=False))
solver.add_pde_term(exahype2.api.MaxEigenvalue(stateless=False))
solver.add_pde_term(exahype2.api.InitPointWisely(stateless=False))

## Numerical scheme and mesh

We stick to Finite Volumes with a Rusanov flux and explicit Euler over
patches of $9 \times 9$ volumes. The domain is five times wider than in
@ref tutorials_exahype2_10_euler. With a maximum volume size of 0.005,
ExaHyPE ends up with $27 \times 27$ patches, i.e. with
$243 \times 243$ volumes, each of size $1/243 \approx 0.004$.

In [ ]:
patch_size  = 9
volume_size = 0.005

solver.numerical_scheme  = exahype2.api.fv.RusanovWithExplicitEuler(solver, patch_size)
solver.grid_construction = exahype2.api.fv.SetupSimulationOverRegularGrid(solver, volume_size=volume_size)

solver.plotter = exahype2.api.fv.PlotAsVTKFile(
    solver,
    first_plot_time_stamp = 0.0,
    time_in_between_plots = 0.01,
)

The picture shows the initial density on this mesh. The white lines
are the patch boundaries. The high-pressure region covers only a handful
of patches:

![Mesh and initial condition](illustrations/13_euler-with-amr-mesh.png)

## Global simulation parameters

We simulate until $t=0.2$. By then, the shock has travelled to a radius of
roughly $0.42$, i.e. it has not reached the boundary yet. The boundary
conditions therefore do not influence the solution, and we can pick the
simplest option: a periodic domain. This way, we do not have to implement
any boundary conditions.

In [ ]:
project.add_solver(solver)

project.set_global_simulation_parameters(
    dimensions   = 2,
    offset       = [-0.5, -0.5],
    size         = [ 1.0,  1.0],
    min_end_time = 0.2,
    max_end_time = 0.2,
    periodic_BC  = [True, True, True],
)

## Lowering into C++

In [ ]:
project.set_Peano4_installation(os.path.abspath("../../"), peano4.api.output.CompileMode.RELEASE)

peano4_project = project.generate_Peano4_project(verbosity=peano4.api.LoggerVerbosity.TEXT)
peano4_project.generate()

## Adding the actual physics

Flux and eigenvalue are exactly the same as in @ref tutorials_exahype2_10_euler,
the physics does not change, only the scenario. The initial condition is
new. It translates the primitive variables $(\rho, u_1, u_2, p)$ from above
into the conserved quantities $Q = (\rho, \rho u_1, \rho u_2, E)$.

**Exercise:** Copy flux() and maxEigenvalue() from your solution of
@ref tutorials_exahype2_10_euler into 13_euler-with-amr/EulerFV.cpp and
implement initPointWisely(). If you want to skip the exercise, run the
cell below. It overwrites the file with our reference implementation.

In [ ]:
%%writefile 13_euler-with-amr/EulerFV.cpp
#include "EulerFV.h"


void tutorials::exahype2::euler::EulerFV::flux(
  const double* __restrict__ Q, const tarch::la::Vector<Dimensions, double>& x, int normal, double* __restrict__ F
) {
  const double irho  = 1.0 / Q[0];
  constexpr double gamma = 1.4;
  const double p = (gamma - 1.0) * (Q[3] - 0.5 * irho * (Q[1] * Q[1] + Q[2] * Q[2]));

  switch (normal) {
    case 0:
      F[0] = Q[1];
      F[1] = irho * Q[1] * Q[1] + p;
      F[2] = irho * Q[1] * Q[2];
      F[3] = irho * Q[1] * (Q[3] + p);
      break;
    case 1:
      F[0] = Q[2];
      F[1] = irho * Q[2] * Q[1];
      F[2] = irho * Q[2] * Q[2] + p;
      F[3] = irho * Q[2] * (Q[3] + p);
      break;
  }
}


void tutorials::exahype2::euler::EulerFV::maxEigenvalue(
  const double* __restrict__ Q, const tarch::la::Vector<Dimensions, double>& x, int normal, double* __restrict__ lambdaMax
) {
  const double irho  = 1.0 / Q[0];
  constexpr double gamma = 1.4;
  const double p = (gamma - 1.0) * (Q[3] - 0.5 * irho * (Q[1] * Q[1] + Q[2] * Q[2]));
  const double c = std::sqrt(gamma * p * irho);

  // velocity, i.e. momentum divided by density
  const double u = Q[1 + normal] * irho;

  lambdaMax[0] = std::max(std::abs(u + c), std::abs(u - c));
}


void tutorials::exahype2::euler::EulerFV::initPointWisely(
  const tarch::la::Vector<Dimensions, double>& x, double* __restrict__ Q
) {
  // Gas at rest. Sod's left state within a circle of radius 0.1 around
  // the origin, Sod's right state outside.
  constexpr double gamma  = 1.4;
  constexpr double radius = 0.1;
  const bool   inside = x[0] * x[0] + x[1] * x[1] < radius * radius;
  const double rho    = inside ? 1.0 : 0.125;
  const double p      = inside ? 1.0 : 0.1;
  Q[0] = rho;
  Q[1] = 0.0;
  Q[2] = 0.0;
  Q[3] = p / (gamma - 1.0);
}

## Log filter, build and run

In [ ]:
%%writefile 13_euler-with-amr/exahype2.log-filter
debug     tarch       -1      black     always-on
debug     peano4      -1      black     always-on
debug     exahype2    -1      black     always-on
debug     toolbox     -1      black     always-on
debug     tutorials   -1      black     always-on

trace     tarch       -1      black     always-on
trace     peano4      -1      black     always-on
trace     exahype2    -1      black     always-on
trace     toolbox     -1      black     always-on
trace     tutorials   -1      black     always-on

info      tarch       -1      white     always-on
info      peano4      -1      white     always-on
info      exahype2    -1      white     always-on
info      toolbox     -1      white     always-on
info      tutorials   -1      white     always-on

In [ ]:
!cd 13_euler-with-amr && make -j

In [ ]:
!cd 13_euler-with-amr && ./euler

The run takes around half a minute and writes 21 snapshots into
13_euler-with-amr/solution-EulerFV.peano-patch-file plus the files it
includes.

## Interactive visualisation

We use the notebook viewer again. As the density drops quickly after the
explosion, the colour range fixed by the first snapshot is not very
helpful here. We therefore pick a narrower range by hand. Play around with
the unknowns: The energy shows the shock very clearly, and the momenta
show that the gas moves outwards.

In [ ]:
import tarch.plotter.api.notebook

tarch.plotter.api.notebook.show_solution(
    "13_euler-with-amr/solution-EulerFV.peano-patch-file",
    unknown       = 0,
    unknown_names = ["rho", "rho u_1", "rho u_2", "E"],
    clim          = (0.05, 0.4),
)

## What we see

The pictures below show the density at three time stamps. Each one uses
its own colour range. The solution stays radially symmetric. At $t=0.05$,
the high density region in the centre has started to expand. At $t=0.1$,
we see a ring of compressed gas behind the shock. At $t=0.2$, the
rarefaction has emptied the centre, and the compressed gas forms a wide
ring between a radius of roughly $0.2$ and the shock at roughly $0.42$.

![Density snapshots](illustrations/13_euler-with-amr-snapshots.png)

The radial profile makes this more explicit. We plot the density along the
positive $x$-axis. The initial jump at $r=0.1$ breaks up into the waves
described above. The shock at the right end of each profile is not a sharp
jump, though: it is smeared over several volumes. This is the
numerical diffusion of the first-order Rusanov scheme that we have already
seen in @ref tutorials_exahype2_10_euler.

![Radial density profile](illustrations/13_euler-with-amr-profile.png)

## Where the regular mesh wastes effort

To get a sharper shock, we have to make the volumes smaller. With a
regular mesh, this is expensive: If we make the volumes three times
smaller, we get nine times as many volumes in 2d. The time step size shrinks
by a factor of three, too, due to the CFL condition. One refinement step
therefore makes the simulation 27 times more expensive in 2d, and 81 times
more expensive in 3d.

Yet, most of the mesh does not need this resolution. The picture below
marks all patches in which the density jumps steeply between neighbouring
volumes at the final time. Only around a quarter of the patches hold such
steep gradients. Furthermore, this set of patches changes over time: early
on, only a few patches in the centre are relevant. The region of interest
then grows as a ring with the shock.

![Patches with steep gradients](illustrations/13_euler-with-amr-active-patches.png)

This is the motivation for adaptive mesh refinement: We want a mesh which
is fine around the waves and coarse elsewhere, and which follows the waves
as they travel through the domain. The regular-mesh solution of this
tutorial serves as reference for such adaptive runs.

**Exercise:** Reduce the volume size to 0.002 and rerun the notebook from
the cell where we set up the mesh. ExaHyPE now uses $81 \times 81$ patches.
Compare the runtime and the sharpness of the shock with the coarse run.

The pictures of this tutorial are created by the script
tutorials/exahype2/illustrations/13_euler-with-amr.py, which runs exactly
this experiment.

## Point-wise adaptive mesh refinement for the initial condition

The initial condition is discontinuous only along the circle of radius 0.1.
A first step towards AMR is therefore to refine the mesh around this circle
while we construct it, i.e. to make the mesh fine where the initial condition
is steep.

For this part of the tutorial, we switch to
exahype2.api.fv.SetupSimulationWithUserCodedRefinementCriterion as the
generator which sets up the simulation. Everything else is the same as for the
regular mesh. We write the code into a separate directory
13_euler-with-amr-initial-refinement, so we can compare the two runs.

In [ ]:
project = exahype2.api.Project(
    namespace    = ["tutorials", "exahype2", "euler"],
    project_name = "Euler",
    directory    = "13_euler-with-amr-initial-refinement",
    executable   = "euler",
)

solver = exahype2.api.Solver(
    "EulerFV",
    4,   # unknowns
    0,   # auxiliary variables
)

solver.add_pde_term(exahype2.api.Flux(stateless=False))
solver.add_pde_term(exahype2.api.MaxEigenvalue(stateless=False))
solver.add_pde_term(exahype2.api.InitPointWisely(stateless=False))

In [ ]:
patch_size  = 9
volume_size = 0.005

solver.numerical_scheme  = exahype2.api.fv.RusanovWithExplicitEuler(solver, patch_size)
solver.grid_construction = exahype2.api.fv.SetupSimulationWithUserCodedRefinementCriterion(solver)

solver.plotter = exahype2.api.fv.PlotAsVTKFile(
    solver,
    first_plot_time_stamp = 0.0,
    time_in_between_plots = 0.01,
)

In [ ]:
project.add_solver(solver)

project.set_global_simulation_parameters(
    dimensions   = 2,
    offset       = [-0.5, -0.5],
    size         = [ 1.0,  1.0],
    min_end_time = 0.2,
    max_end_time = 0.2,
    periodic_BC  = [True, True, True],
)

In [ ]:
project.set_Peano4_installation(os.path.abspath("../../"), peano4.api.output.CompileMode.RELEASE)

peano4_project = project.generate_Peano4_project(verbosity=peano4.api.LoggerVerbosity.TEXT)
peano4_project.generate()

The physics is the same as before, so we write the reference
implementation of flux, eigenvalue and initial condition and the log filter
into the new directory, build and run.

In [ ]:
%%writefile 13_euler-with-amr-initial-refinement/EulerFV.cpp
#include "EulerFV.h"


void tutorials::exahype2::euler::EulerFV::flux(
  const double* __restrict__ Q, const tarch::la::Vector<Dimensions, double>& x, int normal, double* __restrict__ F
) {
  const double irho  = 1.0 / Q[0];
  constexpr double gamma = 1.4;
  const double p = (gamma - 1.0) * (Q[3] - 0.5 * irho * (Q[1] * Q[1] + Q[2] * Q[2]));

  switch (normal) {
    case 0:
      F[0] = Q[1];
      F[1] = irho * Q[1] * Q[1] + p;
      F[2] = irho * Q[1] * Q[2];
      F[3] = irho * Q[1] * (Q[3] + p);
      break;
    case 1:
      F[0] = Q[2];
      F[1] = irho * Q[2] * Q[1];
      F[2] = irho * Q[2] * Q[2] + p;
      F[3] = irho * Q[2] * (Q[3] + p);
      break;
  }
}


void tutorials::exahype2::euler::EulerFV::maxEigenvalue(
  const double* __restrict__ Q, const tarch::la::Vector<Dimensions, double>& x, int normal, double* __restrict__ lambdaMax
) {
  const double irho  = 1.0 / Q[0];
  constexpr double gamma = 1.4;
  const double p = (gamma - 1.0) * (Q[3] - 0.5 * irho * (Q[1] * Q[1] + Q[2] * Q[2]));
  const double c = std::sqrt(gamma * p * irho);

  // velocity, i.e. momentum divided by density
  const double u = Q[1 + normal] * irho;

  lambdaMax[0] = std::max(std::abs(u + c), std::abs(u - c));
}


void tutorials::exahype2::euler::EulerFV::initPointWisely(
  const tarch::la::Vector<Dimensions, double>& x, double* __restrict__ Q
) {
  // Gas at rest. Sod's left state within a circle of radius 0.1 around
  // the origin, Sod's right state outside.
  constexpr double gamma  = 1.4;
  constexpr double radius = 0.1;
  const bool   inside = x[0] * x[0] + x[1] * x[1] < radius * radius;
  const double rho    = inside ? 1.0 : 0.125;
  const double p      = inside ? 1.0 : 0.1;
  Q[0] = rho;
  Q[1] = 0.0;
  Q[2] = 0.0;
  Q[3] = p / (gamma - 1.0);
}

In [ ]:
%%writefile 13_euler-with-amr-initial-refinement/exahype2.log-filter
debug     tarch       -1      black     always-on
debug     peano4      -1      black     always-on
debug     exahype2    -1      black     always-on
debug     toolbox     -1      black     always-on
debug     tutorials   -1      black     always-on

trace     tarch       -1      black     always-on
trace     peano4      -1      black     always-on
trace     exahype2    -1      black     always-on
trace     toolbox     -1      black     always-on
trace     tutorials   -1      black     always-on

info      tarch       -1      white     always-on
info      peano4      -1      white     always-on
info      exahype2    -1      white     always-on
info      toolbox     -1      white     always-on
info      tutorials   -1      white     always-on

In [ ]:
!cd 13_euler-with-amr-initial-refinement && make -j

In [ ]:
!cd 13_euler-with-amr-initial-refinement && ./euler

In [ ]:
import tarch.plotter.api.notebook

tarch.plotter.api.notebook.show_solution(
    "13_euler-with-amr-initial-refinement/solution-EulerFV.peano-patch-file",
    unknown       = 0,
    unknown_names = ["rho", "rho u_1", "rho u_2", "E"],
    clim          = (0.05, 0.4),
)